# 03 – Forecasting Models

Ziel dieses Notebooks ist es, alle Forecast-Modelle aus der Seminararbeit einheitlich zu trainieren, Forecast-Metriken zu vergleichen und die Vorhersagen für Notebook 04 zu speichern.

Verwendete Modelle:

- Persistence 1h und Persistence 24h
- ARIMA
- Linear Regression
- Elastic Net
- Random Forest
- Neural Network
- Quantile Regression
- Quantile Regression Forest

Wichtig: Dieses Notebook erzeugt am Ende `results/forecasts/predictions.csv` und speichert relevante Modelle unter `results/models/`. Notebook 04 sollte später diese Dateien laden und nicht mehr von Variablen aus diesem Kernel abhängen.


## 1. Imports und Pfade

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Projekt-Root finden: Notebook liegt normalerweise in /notebooks
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "final_dataset.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
FORECAST_DIR = RESULTS_DIR / "forecasts"
MODEL_DIR = RESULTS_DIR / "models"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"

for directory in [FORECAST_DIR, MODEL_DIR, FIGURE_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path:    {DATA_PATH}")

## 2. Modellmodule laden

In [ ]:
from src.models import (
    persistence_model,
    arima,
    linear_regression,
    elastic_net,
    random_forest,
    neural_net,
    quantile_regression,
    quantile_regression_forest,
)

print("Model modules loaded.")

## 3. Daten laden und Zielvariable vorbereiten

Für das spätere Bidding sollte die Zielvariable in **MWh** vorliegen, da Strompreise typischerweise in EUR/MWh angegeben werden. Falls `power` in kW vorliegt und stündliche Werte verwendet werden, gilt:

\[
energy\_mwh = \frac{\max(power\_kw, 0)}{1000}
\]

Negative Produktionswerte werden hier für den Einspeise-/Bidding-Benchmark auf 0 begrenzt.


In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])

required_columns = ["timestamp", "power", "wind_speed", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns in final_dataset.csv: {missing_columns}")

# Negative Leistung für den Bidding-Benchmark auf 0 begrenzen
df["power_kw_clipped"] = df["power"].clip(lower=0)

# Stündliche Leistung in kW -> Energie in MWh pro Stunde
df["energy_mwh"] = df["power_kw_clipped"] / 1000.0

FEATURES = ["wind_speed", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]
TARGET = "energy_mwh"

df = df.dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)

print(df.shape)
df[["timestamp", "power", "power_kw_clipped", "energy_mwh"] + FEATURES].head()

## 4. Zeitlicher Train-Test-Split

In [ ]:
TEST_SIZE = 0.2

split_idx = int(len(df) * (1 - TEST_SIZE))

train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

X_train = train_df[FEATURES]
X_test = test_df[FEATURES]

y_train = train_df[TARGET].to_numpy()
y_test = test_df[TARGET].to_numpy()

print(f"Train: {len(train_df)}")
print(f"Test:  {len(test_df)}")
print(f"Features: {FEATURES}")
print(f"Target: {TARGET}")

## 5. Hilfsfunktionen für Evaluation und Plots

In [ ]:
def evaluate_point_forecast(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Compute standard point forecast metrics."""
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }


def pinball_loss(y_true: np.ndarray, y_pred: np.ndarray, quantile: float) -> float:
    """Compute mean pinball loss for one quantile."""
    error = y_true - y_pred
    loss = np.maximum(quantile * error, (quantile - 1) * error)
    return float(np.mean(loss))


def plot_forecast_vs_actual(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    title: str,
    filename: str,
    n_points: int = 250,
) -> None:
    """Plot actual vs. predicted values and save the figure."""
    n = min(n_points, len(y_true))

    plt.figure(figsize=(12, 4))
    plt.plot(y_true[:n], label="Actual", linewidth=1)
    plt.plot(y_pred[:n], label="Forecast", linewidth=1)
    plt.title(title)
    plt.xlabel("Test observation")
    plt.ylabel("Energy [MWh]")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()


def plot_quantile_forecast(
    y_true: np.ndarray,
    q_low: np.ndarray,
    q_med: np.ndarray,
    q_high: np.ndarray,
    title: str,
    filename: str,
    n_points: int = 250,
) -> None:
    """Plot quantile forecast interval and save the figure."""
    n = min(n_points, len(y_true))
    x = np.arange(n)

    plt.figure(figsize=(12, 4))
    plt.plot(x, y_true[:n], label="Actual", linewidth=1)
    plt.plot(x, q_med[:n], label="Median forecast", linewidth=1)
    plt.fill_between(x, q_low[:n], q_high[:n], alpha=0.25, label="Prediction interval")
    plt.title(title)
    plt.xlabel("Test observation")
    plt.ylabel("Energy [MWh]")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()

## 6. Persistence Baselines

In [ ]:
# Persistence 1h: y_hat(t) = y(t-1)
pm_model = persistence_model.train(X_train, y_train, lag=1)
y_pred_pm = persistence_model.predict(pm_model, X_test, y_test)

# Persistence 24h: y_hat(t) = y(t-24)
pm24_model = persistence_model.train(X_train, y_train, lag=24)
y_pred_pm24 = persistence_model.predict(pm24_model, X_test, y_test)

plot_forecast_vs_actual(y_test, y_pred_pm, "Persistence Forecast (1h)", "forecast_persistence_1h.png")
plot_forecast_vs_actual(y_test, y_pred_pm24, "Persistence Forecast (24h)", "forecast_persistence_24h.png")

## 7. ARIMA

In [ ]:
arima_model = arima.train(y_train)
y_pred_arima = arima.predict(arima_model, steps=len(y_test))
y_pred_arima = np.clip(y_pred_arima, 0, None)

plot_forecast_vs_actual(y_test, y_pred_arima, "ARIMA Forecast", "forecast_arima.png")

## 8. Linear Regression

In [ ]:
lr_model = linear_regression.train(X_train, y_train)
y_pred_lr = linear_regression.predict(lr_model, X_test)

plot_forecast_vs_actual(y_test, y_pred_lr, "Linear Regression Forecast", "forecast_linear_regression.png")

## 9. Elastic Net

In [ ]:
en_model = elastic_net.train(
    X_train,
    y_train,
    alpha=0.1,
    l1_ratio=0.5,
)

y_pred_en = elastic_net.predict(en_model, X_test)

plot_forecast_vs_actual(y_test, y_pred_en, "Elastic Net Forecast", "forecast_elastic_net.png")

## 10. Random Forest

In [ ]:
rf_model = random_forest.train(
    X_train,
    y_train,
    n_estimators=300,
    min_samples_leaf=2,
)

y_pred_rf = random_forest.predict(rf_model, X_test)

plot_forecast_vs_actual(y_test, y_pred_rf, "Random Forest Forecast", "forecast_random_forest.png")

## 11. Neural Network

In [ ]:
nn_model = neural_net.train(
    X_train,
    y_train,
    hidden_layer_sizes=(64, 32),
    max_iter=1000,
)

y_pred_nn = neural_net.predict(nn_model, X_test)

plot_forecast_vs_actual(y_test, y_pred_nn, "Neural Network Forecast", "forecast_neural_network.png")

## 12. Quantile Regression

In [ ]:
QUANTILES = (0.25, 0.5, 0.75)

qr_models = quantile_regression.train(
    X_train,
    y_train,
    quantiles=QUANTILES,
)

qr_predictions = quantile_regression.predict_all(qr_models, X_test)

y_pred_qr_q25 = qr_predictions["q25"].to_numpy()
y_pred_qr_q50 = qr_predictions["q50"].to_numpy()
y_pred_qr_q75 = qr_predictions["q75"].to_numpy()

plot_quantile_forecast(
    y_test,
    y_pred_qr_q25,
    y_pred_qr_q50,
    y_pred_qr_q75,
    "Quantile Regression Forecast",
    "forecast_quantile_regression.png",
)

## 13. Quantile Regression Forest

In [ ]:
qrf_model = quantile_regression_forest.train(
    X_train,
    y_train,
    n_estimators=300,
    min_samples_leaf=5,
)

qrf_predictions = quantile_regression_forest.predict_all(
    qrf_model,
    X_test,
    quantiles=QUANTILES,
)

y_pred_qrf_q25 = qrf_predictions["q25"].to_numpy()
y_pred_qrf_q50 = qrf_predictions["q50"].to_numpy()
y_pred_qrf_q75 = qrf_predictions["q75"].to_numpy()

plot_quantile_forecast(
    y_test,
    y_pred_qrf_q25,
    y_pred_qrf_q50,
    y_pred_qrf_q75,
    "Quantile Regression Forest Forecast",
    "forecast_quantile_regression_forest.png",
)

## 14. Forecast-Metriken vergleichen

In [ ]:
point_predictions = {
    "Persistence_1h": y_pred_pm,
    "Persistence_24h": y_pred_pm24,
    "ARIMA": y_pred_arima,
    "Linear_Regression": y_pred_lr,
    "Elastic_Net": y_pred_en,
    "Random_Forest": y_pred_rf,
    "Neural_Network": y_pred_nn,
    "Quantile_Regression_q50": y_pred_qr_q50,
    "QRF_q50": y_pred_qrf_q50,
}

metric_rows = []

for model_name, y_pred in point_predictions.items():
    row = {"model": model_name}
    row.update(evaluate_point_forecast(y_test, y_pred))
    metric_rows.append(row)

# Pinball Loss für probabilistische Modelle ergänzen
pinball_rows = []
for q in QUANTILES:
    pinball_rows.append({
        "model": "Quantile_Regression",
        "quantile": q,
        "pinball_loss": pinball_loss(y_test, qr_predictions[f"q{int(q*100):02d}"].to_numpy(), q),
    })
    pinball_rows.append({
        "model": "QRF",
        "quantile": q,
        "pinball_loss": pinball_loss(y_test, qrf_predictions[f"q{int(q*100):02d}"].to_numpy(), q),
    })

df_metrics = pd.DataFrame(metric_rows).sort_values("rmse")
df_pinball = pd.DataFrame(pinball_rows)

display(df_metrics)
display(df_pinball)

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(df_metrics["model"], df_metrics["rmse"])
plt.title("Forecast comparison by RMSE")
plt.ylabel("RMSE [MWh]")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "metric_comparison_rmse.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 4))
plt.bar(df_metrics["model"], df_metrics["mae"])
plt.title("Forecast comparison by MAE")
plt.ylabel("MAE [MWh]")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "metric_comparison_mae.png", dpi=150, bbox_inches="tight")
plt.show()

## 15. Ergebnisse speichern

Die gespeicherten Dateien sind die Schnittstelle zu Notebook 04.

- `results/forecasts/predictions.csv`: tatsächliche Werte und alle Forecasts
- `results/forecasts/test_features.csv`: Test-Features
- `results/tables/forecast_metrics.csv`: klassische Forecast-Metriken
- `results/tables/pinball_metrics.csv`: Pinball Loss für Quantilmodelle
- `results/models/*.joblib`: trainierte Modelle


In [ ]:
predictions = pd.DataFrame({
    "timestamp": test_df["timestamp"].to_numpy(),
    "y_true_mwh": y_test,
    "persistence_1h": y_pred_pm,
    "persistence_24h": y_pred_pm24,
    "arima": y_pred_arima,
    "linear_regression": y_pred_lr,
    "elastic_net": y_pred_en,
    "random_forest": y_pred_rf,
    "neural_network": y_pred_nn,
    "quantile_regression_q25": y_pred_qr_q25,
    "quantile_regression_q50": y_pred_qr_q50,
    "quantile_regression_q75": y_pred_qr_q75,
    "qrf_q25": y_pred_qrf_q25,
    "qrf_q50": y_pred_qrf_q50,
    "qrf_q75": y_pred_qrf_q75,
})

predictions.to_csv(FORECAST_DIR / "predictions.csv", index=False)
test_df[["timestamp"] + FEATURES].to_csv(FORECAST_DIR / "test_features.csv", index=False)

df_metrics.to_csv(TABLE_DIR / "forecast_metrics.csv", index=False)
df_pinball.to_csv(TABLE_DIR / "pinball_metrics.csv", index=False)

joblib.dump(pm_model, MODEL_DIR / "persistence_1h.joblib")
joblib.dump(pm24_model, MODEL_DIR / "persistence_24h.joblib")
joblib.dump(arima_model, MODEL_DIR / "arima.joblib")
joblib.dump(lr_model, MODEL_DIR / "linear_regression.joblib")
joblib.dump(en_model, MODEL_DIR / "elastic_net.joblib")
joblib.dump(rf_model, MODEL_DIR / "random_forest.joblib")
joblib.dump(nn_model, MODEL_DIR / "neural_network.joblib")
joblib.dump(qr_models, MODEL_DIR / "quantile_regression.joblib")
joblib.dump(qrf_model, MODEL_DIR / "quantile_regression_forest.joblib")

print(f"Saved predictions to: {FORECAST_DIR / 'predictions.csv'}")
print(f"Saved metrics to:     {TABLE_DIR / 'forecast_metrics.csv'}")
print(f"Saved models to:      {MODEL_DIR}")

## 16. Key Insight

Das Forecast-Ranking nach RMSE oder MAE ist nur die erste Bewertung. Für die Seminarfrage ist entscheidend, ob diese Prognosen im Newsvendor-basierten Bidding auch wirtschaftlich gute Gebote erzeugen.

Weiter geht es in Notebook 04 mit:

1. Laden von `results/forecasts/predictions.csv`
2. Definition von Kostenszenarien
3. Berechnung von Newsvendor-Loss und Regret
4. Vergleich von Point Forecasts und probabilistischen Forecasts
